# 02 — Clean Silver Layer

Load bronze stock price data, apply cleaning transformations, compute metrics, and save to the silver layer.

**Silver columns:** `date`, `ticker`, `open`, `high`, `low`, `close`, `adj_close`, `volume`, `daily_return`, `dollar_volume`, `source`, `loaded_at`

**Flow:** bronze CSV → `clean_stock_data` → inspect → save silver CSV

**Prerequisite:** Run `01_ingest_bronze.ipynb` first so `data/processed/bronze/bronze_stock_prices.csv` exists.

## Setup

Add `src` to the Python path and import the silver transformation helper.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
BRONZE_INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "bronze" / "bronze_stock_prices.csv"
SILVER_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "silver" / "silver_stock_prices.csv"

sys.path.insert(0, str(SRC_DIR))

from transformations import clean_stock_data

print(f"Project root: {PROJECT_ROOT}")
print(f"Bronze input: {BRONZE_INPUT_PATH}")
print(f"Silver output: {SILVER_OUTPUT_PATH}")

## Load bronze data

Read the combined bronze file created by the ingest notebook.

In [ ]:
if not BRONZE_INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Bronze file not found at {BRONZE_INPUT_PATH}. Run 01_ingest_bronze.ipynb first."
    )

bronze_df = pd.read_csv(BRONZE_INPUT_PATH)

print(f"Bronze rows: {len(bronze_df):,}")
print(f"Tickers: {sorted(bronze_df['ticker'].unique())}")
bronze_df.head()

## Clean and transform

`clean_stock_data` runs the full silver pipeline:

- Standardize column names
- Convert data types
- Remove duplicates
- Handle missing values
- Sort by ticker and date
- Calculate `daily_return` and `dollar_volume`

In [ ]:
silver_df = clean_stock_data(bronze_df)

print(f"Silver rows: {len(silver_df):,}")
print(f"Silver columns: {list(silver_df.columns)}")
silver_df.head(10)

## Inspect silver data

Check data types and preview the computed metrics.

In [ ]:
silver_df.info()

In [ ]:
silver_df[["date", "ticker", "close", "daily_return", "volume", "dollar_volume"]].head(10)

## Save silver layer

Write the cleaned data to `data/processed/silver/`.

In [ ]:
SILVER_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
silver_df.to_csv(SILVER_OUTPUT_PATH, index=False)

print(f"Saved silver data to {SILVER_OUTPUT_PATH}")